In [1]:
import sys
sys.path.append('../externals/DynaMix-python')

import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from neurodsp.spectral import compute_spectrum
from neurodsp.plts.spectral import plot_power_spectra
import os
from joblib import Parallel, delayed
from tqdm import tqdm

from src.dynamix.model.forecaster import DynaMixForecaster
from src.dynamix.utilities.plotting_eval import plot_TS_forecast, plot_3D_attractor
from src.dynamix.utilities.utilities import load_hf_model

from sklearn.metrics import make_scorer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, StratifiedShuffleSplit
from sklearn.model_selection import permutation_test_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import pickle
import os
from collections import Counter

import torch

In [2]:
# define file paths
eeg_path = '/oscar/data/sjones/shared/TDBRAIN_preprocessed/preprocessed'
metadata_path = '../data/TDBRAIN_participants_V2.tsv'
subj_list = os.listdir(eeg_path)
print("Number of total subjects:", len(subj_list))

# read data
metadata_df = pd.read_csv(metadata_path, delimiter='\t')

# generate mask to pull out MDD, DISC subjects only
subj_mask = np.isin(metadata_df['participants_ID'].values, subj_list)
discovery_mask = metadata_df['DISC/REP'].values == 'DISCOVERY'
dataset_mask = metadata_df['Dataset'].values == 'MDD-rTMS'
rTMS_mask = ~metadata_df['rTMS PROTOCOL'].isna() # excludes 1 participant

mask = np.logical_and.reduce([subj_mask, discovery_mask, dataset_mask, rTMS_mask])

df = metadata_df[mask].copy() # .copy because plan to add columns later
df['age']  = df['age'].str.replace(',', '.').astype(float) # convert age to float
print("Shape after keeping MDD-rTMS, Discovery rows:", df.shape)

duplicate_ids = df["participants_ID"].value_counts()
duplicate_ids = duplicate_ids[duplicate_ids > 1].index
df = df.drop_duplicates(subset="participants_ID", keep="first")
print("Shape after keeping only first entry for each participant:", df.shape)

# save paths to eeg numpy files for each subject in df
ec_eeg_path, eo_eeg_path, has_ses1 = list(), list(), list()

for subj_id in df['participants_ID'].values:
    subj_path = f'{eeg_path}/{subj_id}/ses-1/eeg'
    if os.path.isdir(subj_path):
        has_ses1.append(True)
        subj_files = os.listdir(subj_path)
        ec_subj_files = list(pathlib.Path(subj_path).glob('*restEC*.npy'))
        eo_subj_files = list(pathlib.Path(subj_path).glob('*restEO*.npy'))
        assert len(ec_subj_files) == len(eo_subj_files) == 1

        ec_eeg_path.append(str(ec_subj_files[0]))
        eo_eeg_path.append(str(eo_subj_files[0]))

    else:
        has_ses1.append(False)
        ec_eeg_path.append('')
        eo_eeg_path.append('')

df['ec_eeg_path'] = ec_eeg_path
df['eo_eeg_path'] = eo_eeg_path
df['has_ses1'] = has_ses1

df = df[df['has_ses1'] == True].reset_index(drop=True)
print("Shape after removing subj with no session 1 data:", df.shape)

# load the pre-trained model
model = load_hf_model("dynamix-6d-alrnn-v1.0")

# set model to evaluation mode
model.eval()

# initialize the forecaster
forecaster = DynaMixForecaster(model)

Number of total subjects: 1274
Shape after keeping MDD-rTMS, Discovery rows: (131, 111)
Shape after keeping only first entry for each participant: (123, 111)
Shape after removing subj with no session 1 data: (120, 114)


In [3]:
def get_dynamix_latent(subj_data_path):

    channel_filter = ['F7', 'P8', 'T7', 'O2']
    # channel_filter = ['Fp1', 'Fz', 'F4', 'O1', 'Oz', 'O2',]

    eeg_dict = np.load(subj_data_path, allow_pickle=True)
    channel_labels = eeg_dict['labels']
    sampling_freq = eeg_dict['Fs']

    channel_mask = np.isin(channel_labels, channel_filter)
    eeg_data = eeg_dict['data'][0, channel_mask, :]
    assert eeg_data.shape[0] == len(channel_filter)

    offset = 5000
    CL = 10000
    T = 100

    context_start = offset
    context_end = offset + CL

    # Load the time series data
    ts_data = eeg_data.T

    context_ts = ts_data[context_start:context_end,:] # context from 5000 to 15000 (10s-30s)

    # Convert to PyTorch tensor
    context_ts_tensor = torch.tensor(context_ts, dtype=torch.float32)
    
    activation = {}
    # a dict to store the activations
    def getActivation(name):
        activation[name] = list()
        # the hook signature
        def hook(model, input, output):
            activation[name].append(output.detach())
        return hook
    
    hooks = [
        forecaster.model.gating_network.register_forward_hook(getActivation("w_exp")),
        forecaster.model.gating_network.mlp_layer1.register_forward_hook(getActivation("mlp1")),
        forecaster.model.gating_network.mlp_layer2.register_forward_hook(getActivation("mlp2")),
        forecaster.model.register_forward_hook(getActivation("model_out")),
    ]

    # h = forecaster.model.gating_network.register_forward_hook(getActivation('w_exp'))

    # Make prediction
    with torch.no_grad():  # No gradient tracking needed for inference
        reconstruction_ts = forecaster.forecast(
            context=context_ts_tensor,
            horizon=T,
            preprocessing_method="pos_embedding",
            standardize=True,
            fit_nonstationary=False,
        )

    # h.remove()
    for h in hooks:
        h.remove()

    w_exp = torch.stack(activation["w_exp"]).numpy().squeeze()         # (T, 80)
    mlp1 = torch.stack(activation["mlp1"]).numpy().squeeze()           # (T, 80)
    mlp2 = torch.stack(activation["mlp2"]).numpy().squeeze()           # (T, 80)
    model_out = torch.stack(activation["model_out"]).numpy().squeeze() # (T, M)

    feats = np.concatenate([
        np.std(w_exp, axis=0),
        np.mean(w_exp, axis=0),
        np.std(mlp1, axis=0),
        np.mean(mlp1, axis=0),
        np.std(mlp2, axis=0),
        np.mean(mlp2, axis=0),
        np.std(model_out, axis=0),
        np.mean(model_out, axis=0),
    ])

    return feats

# extract features in parallel for all subjects
ec_res = Parallel(n_jobs=16)(delayed(get_dynamix_latent)(ec_path) for ec_path in tqdm(df['ec_eeg_path'].values))
eo_res = Parallel(n_jobs=16)(delayed(get_dynamix_latent)(ec_path) for ec_path in tqdm(df['eo_eeg_path'].values))

ec_features = [res[0] for res in ec_res]
eo_features = [res[0] for res in eo_res]

ec_features = np.array(ec_features)
eo_features = np.array(eo_features)

M = ec_res[0].shape[0] - 80*6
M = M // 2
print("M:", M)

col_names = (
    [f"dynamix_wexp_std_{i+1}" for i in range(80)] +
    [f"dynamix_wexp_mean_{i+1}" for i in range(80)] +
    [f"dynamix_mlp1_std_{i+1}" for i in range(80)] +
    [f"dynamix_mlp1_mean_{i+1}" for i in range(80)] +
    [f"dynamix_mlp2_std_{i+1}" for i in range(80)] +
    [f"dynamix_mlp2_mean_{i+1}" for i in range(80)] +
    [f"dynamix_modelout_std_{i+1}" for i in range(M)] +
    [f"dynamix_modelout_mean_{i+1}" for i in range(M)]
)

df_dynamix = pd.DataFrame(np.vstack(ec_res), columns=col_names)
df_dynamix.insert(0, "participants_ID", df['participants_ID'].values)


  0%|          | 0/120 [00:00<?, ?it/s]

100%|██████████| 120/120 [00:05<00:00, 23.51it/s]


M: 10


In [4]:
response_var = "remitter"
response_var_cap = "Remitter"
conditions = "mdd_dyna_wexp"
conditions_cap = "MDD Dynamix Expert Weights"
eval_metric = "nppv"
eval_metric_cap = "nPPV"

df = pd.merge(df_dynamix, df, on='participants_ID', how='inner')
print("Merged dataframe shape:", df.shape)
print(df.head())

Merged dataframe shape: (120, 614)
  participants_ID  dynamix_wexp_std_1  dynamix_wexp_std_2  dynamix_wexp_std_3  \
0    sub-87999321        3.526916e-07            0.007617            0.035005   
1    sub-88000181        1.747906e-07            0.004402            0.018313   
2    sub-88000313        4.025473e-07            0.005048            0.020056   
3    sub-88000489        7.630833e-06            0.005419            0.021625   
4    sub-88000533        3.711383e-07            0.005285            0.021090   

   dynamix_wexp_std_4  dynamix_wexp_std_5  dynamix_wexp_std_6  \
0            0.009739            0.010505        4.554098e-07   
1            0.007793            0.004139        1.878319e-07   
2            0.010556            0.004936        5.003621e-07   
3            0.005107            0.002417        9.656854e-06   
4            0.024361            0.011642        4.598213e-07   

   dynamix_wexp_std_7  dynamix_wexp_std_8  dynamix_wexp_std_9  ...  \
0            0.01

In [5]:
cols = ['age', 'gender', 'BDI_pre']
# dynamix_cols = df.filter(regex='dynamix_').columns.tolist()
dynamix_cols = df.filter(regex='dynamix_wexp_std').columns.tolist()
# print(dynamix_cols)

# construct X
X = df[cols + dynamix_cols]
# X = df[cols]
y = df[response_var_cap].astype(float)

# binary vars float --> str
X = X.copy()
X['gender'] = X['gender'].map({1.0: 'male', 0.0: 'female'})

print("X shape:", X.shape)
print("X columns:", X.columns)
# print("X column datatypes:", X.dtypes)
print(X.head())

# reduce feature dimensions by removing highly corr columns
def remove_highly_correlated_columns(df, threshold=0.95, exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = []

    feature_cols = [c for c in df.columns if c not in exclude_cols]
    corr = df[feature_cols].corr().abs()

    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    cols_to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

    df_reduced = df.drop(columns=cols_to_drop)
    return df_reduced, cols_to_drop

X, dropped_cols = remove_highly_correlated_columns(
    X,
    threshold=0.5,
    exclude_cols=["participants_ID", "gender"]
)

print("Dropped:", len(dropped_cols))
print(dropped_cols)
print("New shape:", X.shape)

# construct preprocesser
# decide which encoder to use on each feature
dynamix_cols = X.filter(regex='dynamix_').columns.tolist()


onehot_ftrs = ['gender']
std_ftrs = ['age', 'BDI_pre'] + dynamix_cols

# one hot pipeline
# X['ever_used_drugs'] = X['ever_used_drugs'].fillna(99)
onehot_transformer = Pipeline(steps=[
    ('imputer0', SimpleImputer(strategy='constant')), 
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
])

# numeric pipeline with imputation + scaling
numeric_transformer = Pipeline(steps=[
    ('imputer2', SimpleImputer(strategy='median')),  # fills missing values with median
    ('scaler', StandardScaler())                     # scales features
])

# collect all the encoders
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_transformer, onehot_ftrs), 
        ('std', numeric_transformer, std_ftrs)])

clf = Pipeline(steps=[('preprocessor', preprocessor)])

# construct custom nppv scoring function
def nppv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    denom_ppv = tp + fp
    percent_pos = np.mean(y_true == 1)

    if denom_ppv == 0 or percent_pos == 0:
        return 0
        
    ppv = tp / denom_ppv
    return (ppv / percent_pos) * 100 # can do -1 here too

nppv_scorer = make_scorer(nppv)

X shape: (120, 83)
X columns: Index(['age', 'gender', 'BDI_pre', 'dynamix_wexp_std_1', 'dynamix_wexp_std_2',
       'dynamix_wexp_std_3', 'dynamix_wexp_std_4', 'dynamix_wexp_std_5',
       'dynamix_wexp_std_6', 'dynamix_wexp_std_7', 'dynamix_wexp_std_8',
       'dynamix_wexp_std_9', 'dynamix_wexp_std_10', 'dynamix_wexp_std_11',
       'dynamix_wexp_std_12', 'dynamix_wexp_std_13', 'dynamix_wexp_std_14',
       'dynamix_wexp_std_15', 'dynamix_wexp_std_16', 'dynamix_wexp_std_17',
       'dynamix_wexp_std_18', 'dynamix_wexp_std_19', 'dynamix_wexp_std_20',
       'dynamix_wexp_std_21', 'dynamix_wexp_std_22', 'dynamix_wexp_std_23',
       'dynamix_wexp_std_24', 'dynamix_wexp_std_25', 'dynamix_wexp_std_26',
       'dynamix_wexp_std_27', 'dynamix_wexp_std_28', 'dynamix_wexp_std_29',
       'dynamix_wexp_std_30', 'dynamix_wexp_std_31', 'dynamix_wexp_std_32',
       'dynamix_wexp_std_33', 'dynamix_wexp_std_34', 'dynamix_wexp_std_35',
       'dynamix_wexp_std_36', 'dynamix_wexp_std_37', 'dynamix_

In [6]:
def MLpipe_SKFold(X, y, preprocessor, ML_algo, param_grid, model_name, eval_metric, num_states=5, do_permutation_test=False):
    '''
    Pipeline with:
        Splitting method -- StratifiedKFold
        Evaluation metric -- Depends on input
    '''

    train_scores = []
    test_scores = []
    best_models = []
    cv_scores = []
    perm_scores_list = []
    perm_pvalues = []
    baseline_scores = []

    print(f"\n=== Training {model_name} ({eval_metric}) ===")

    for i in range(num_states):
        # define random state
        random_state = 29*(i+1)
        
        # --- SPLIT ---
        X_other, X_test, y_other, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=random_state) 
        majority_class = Counter(y_test).most_common(1)[0][1]
        baseline_acc = majority_class / len(y_test)
        baseline_scores.append(baseline_acc)

        # construct kfold object
        kf = StratifiedKFold(n_splits=4, shuffle=True, random_state=random_state)

        # --- PREPROCESS---
        # main preprocessor is fed into function as "preprocessor"
        final_scaler = StandardScaler()

        # --- PIPELINE & GRID SEARCH ---
        pipeline = make_pipeline(preprocessor, ML_algo)
        if eval_metric == "nppv":
            scoring_func = nppv_scorer
        else:
            scoring_func = eval_metric
        grid = GridSearchCV(pipeline, param_grid=param_grid, cv=kf, scoring=scoring_func, return_train_score=True, n_jobs=-1, verbose=0)
        grid.fit(X_other, y_other)
        results = pd.DataFrame(grid.cv_results_)
        # print("\nGrid search results:\n", results)

        # --- BEST MODEL & SCORES ---
        best_model = grid.best_estimator_
        best_models.append(best_model)
        # print(f"Random state {random_state}: {grid.best_params_}")
        
        # train score
        train_score = grid.cv_results_['mean_train_score'][grid.best_index_]
        train_scores.append(train_score)

        # test score
        y_test_pred = best_model.predict(X_test)
        if eval_metric == "accuracy":
            best_test_score = accuracy_score(y_test, y_test_pred)
        elif eval_metric == "precision":
            best_test_score = precision_score(y_test, y_test_pred, average="macro")
        elif eval_metric == "f1_macro":
            best_test_score = f1_score(y_test, y_test_pred, average="macro")
        elif eval_metric == "f1_weighted":
            best_test_score = f1_score(y_test, y_test_pred, average="weighted")
        elif eval_metric == "nppv":
            best_test_score = nppv(y_test, y_test_pred)
            tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()

        else:
            raise ValueError("Evaluation metric not handled in this pipeline.") 
        test_scores.append(best_test_score)
       
        # --- OPTIONAL PERMUTATION TEST ---
        if do_permutation_test:
            cv_score, perm_scores, p_value = permutation_test_score(
                best_model,
                X_other,
                y_other,
                scoring=scoring_func,
                cv=kf,
                n_permutations=1000,
                random_state=random_state,
                n_jobs=-1
            )
            # cv_scores.append(cv_score)
            # perm_scores_list.append(perm_scores)
            # perm_pvalues.append(p_value)
            # print(f"    Permutation CV score: {cv_score:.4f} | p-value: {p_value:.4f}")

    # compute mean baseline
    mean_baseline = np.mean(baseline_scores)
        
    # --- SAVE RESULTS ---
    # file_path = os.path.join(f'../../results/{conditions}/{model_name}_{eval_metric}_{response_var}.save')
    # with open(file_path, 'wb') as file:
    #     pickle.dump((best_models, test_scores), file)

    return (
        train_scores,
        test_scores,
        best_models,
        mean_baseline,
        cv_scores if do_permutation_test else None,
        perm_scores_list if do_permutation_test else None,
        perm_pvalues if do_permutation_test else None
        ,
    )

    # define model algorithms and parameter grids
models = {
    'SimpleLogisticRegression': (
        LogisticRegression(solver='saga', class_weight='balanced', max_iter=1000000), {} 
    ),
    'L1LogisticRegression': (
        LogisticRegression(solver='saga', class_weight='balanced', max_iter=1000000),
        {'logisticregression__C': [0.01, 0.1, 1, 10, 100],
         'logisticregression__l1_ratio': [1]}
    ),
    'L2LogisticRegression': (
        LogisticRegression(solver='saga', class_weight='balanced', max_iter=1000000),
        {'logisticregression__C': [0.0001, 0.001, 0.01, 0.1, 1],
         'logisticregression__l1_ratio': [0]}
    ),
    'ElasticNet': (
        LogisticRegression(solver='saga', class_weight='balanced', max_iter=100000000),
        {'logisticregression__C': [0.0001, 0.001, 0.01, 0.1, 1],
         'logisticregression__l1_ratio': [0.0001, 0.001, 0.01, 0.1, 1]}
    ),
    'SupportVectorClassifier': (
        SVC(probability=True),
        {'svc__C': [0.0001, 0.001, 0.01, 0.1, 1],
         'svc__gamma': [0.001, 0.01, 0.1],
         'svc__kernel': ['rbf']
        }
    ),
}

In [7]:
# evaluation metric = nppv
summary_acc = {}
perm_plot_data = {}
n_random_states = 100
for model_name, (algo, param_grid) in models.items():
    train_scores, test_scores, best_models, mean_baseline, cv_scores, perm_scores_list, perm_pvalues = MLpipe_SKFold(X, y, preprocessor, algo, param_grid, model_name, eval_metric, num_states=n_random_states)

    # clean print of scores
    train_scores_clean = [round(float(s), 4) for s in train_scores]
    test_scores_clean = [round(float(s), 4) for s in test_scores]
    # cv_scores_clean = [round(float(s), 4) for s in cv_scores]
    # perm_pvalues_clean = [round(float(p), 4) for p in perm_pvalues]

    print(f"Train Score: {np.mean(train_scores_clean):.2f} ± {np.std(train_scores_clean):.3f}")
    print(f"Test Score: {np.mean(test_scores_clean):.2f} ± {np.std(test_scores_clean):.3f}")

    summary_acc[model_name] = (
        np.mean(train_scores),
        np.mean(test_scores),
        np.std(test_scores),
        # np.mean(cv_scores),
        # np.mean([np.mean(p) for p in perm_scores_list]),
        # np.std([np.mean(p) for p in perm_scores_list]),
        # np.mean(perm_pvalues)
    )

    perm_plot_data[model_name] = (cv_scores, perm_scores_list, perm_pvalues)


=== Training SimpleLogisticRegression (nppv) ===
Train Score: 143.78 ± 3.056
Test Score: 139.35 ± 30.405

=== Training L1LogisticRegression (nppv) ===
Train Score: 139.32 ± 7.512
Test Score: 134.98 ± 31.370

=== Training L2LogisticRegression (nppv) ===
Train Score: 144.06 ± 6.822
Test Score: 139.06 ± 31.783

=== Training ElasticNet (nppv) ===
Train Score: 139.96 ± 9.125
Test Score: 133.62 ± 33.739

=== Training SupportVectorClassifier (nppv) ===
Train Score: 137.62 ± 11.256
Test Score: 122.24 ± 27.050
